# ⚙️ Middleware

**Add cross-cutting concerns to your API**

---

## 📋 Overview

**What you'll learn:**
- Middleware concepts
- Logging middleware
- CORS configuration
- Compression
- Request/response timing

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from fastapi import FastAPI, Request, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
from fastapi.middleware.trustedhost import TrustedHostMiddleware
import time
import logging
import uuid

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ Setup complete")

## 🤔 What is Middleware?

### Middleware Pattern:

```
Request Flow:

Client
  ↓
  Middleware 1  (CORS)
  ↓
  Middleware 2  (Logging)
  ↓
  Middleware 3  (Auth)
  ↓
  Endpoint Handler
  ↓
  Middleware 3  (Timing)
  ↓
  Middleware 2  (Compression)
  ↓
  Middleware 1  (Headers)
  ↓
Client
```

### Common Use Cases:

- **Logging** - Log all requests/responses
- **CORS** - Handle cross-origin requests
- **Authentication** - Verify tokens
- **Rate limiting** - Throttle requests
- **Compression** - Compress responses
- **Timing** - Track request duration
- **Error handling** - Catch exceptions
- **Request ID** - Track requests

## 📝 Logging Middleware

In [ ]:
from fastapi import FastAPI, Request
import time
import logging

app = FastAPI()
logger = logging.getLogger(__name__)

@app.middleware("http")
async def log_requests(request: Request, call_next):
    """Log all requests and responses."""
    
    start_time = time.time()
    
    # Log request
    logger.info(
        f"Request: {request.method} {request.url.path} "
        f"from {request.client.host}"
    )
    
    # Process request
    response = await call_next(request)
    
    # Calculate duration
    duration = time.time() - start_time
    
    # Log response
    logger.info(
        f"Response: {response.status_code} "
        f"in {duration*1000:.2f}ms"
    )
    
    return response

@app.get("/test")
async def test():
    return {"message": "Hello"}

print("📝 Logging Middleware")
print("\nExample logs:")
print("  Request: GET /test from 127.0.0.1")
print("  Response: 200 in 15.23ms")

## 🆔 Request ID Middleware

In [ ]:
import uuid
from fastapi import Request, Response

@app.middleware("http")
async def add_request_id(request: Request, call_next):
    """Add unique request ID to each request."""
    
    # Generate or get existing request ID
    request_id = request.headers.get('X-Request-ID', str(uuid.uuid4()))
    
    # Store in request state
    request.state.request_id = request_id
    
    # Process request
    response = await call_next(request)
    
    # Add to response headers
    response.headers['X-Request-ID'] = request_id
    
    return response

# Use in logging
@app.middleware("http")
async def log_with_request_id(request: Request, call_next):
    """Log with request ID."""
    
    request_id = getattr(request.state, 'request_id', 'unknown')
    
    logger.info(
        f"[{request_id}] {request.method} {request.url.path}"
    )
    
    response = await call_next(request)
    
    logger.info(
        f"[{request_id}] {response.status_code}"
    )
    
    return response

print("🆔 Request ID Middleware")
print("\nExample:")
print("  [a1b2c3d4] GET /api/chat")
print("  [a1b2c3d4] 200")
print("\n💡 Helps trace requests across services")

## ⏱️ Timing Middleware

In [ ]:
import time
from fastapi import Request

@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    """Add processing time to response headers."""
    
    start_time = time.time()
    
    # Process request
    response = await call_next(request)
    
    # Calculate time
    process_time = time.time() - start_time
    
    # Add headers
    response.headers['X-Process-Time'] = f"{process_time:.4f}"
    response.headers['X-Process-Time-Ms'] = f"{process_time*1000:.2f}"
    
    # Alert on slow requests
    if process_time > 1.0:
        logger.warning(
            f"Slow request: {request.url.path} "
            f"took {process_time:.2f}s"
        )
    
    return response

print("⏱️ Timing Middleware")
print("\nResponse headers:")
print("  X-Process-Time: 0.1234")
print("  X-Process-Time-Ms: 123.40")

## 🌐 CORS Middleware

In [ ]:
print("""
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

# Development - Allow all
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # ⚠️ DON'T use in production!
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Production - Specific origins
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "https://yourdomain.com",
        "https://app.yourdomain.com",
    ],
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["Authorization", "Content-Type"],
    max_age=3600,  # Cache preflight for 1 hour
)

# Environment-based CORS
import os

if os.getenv("ENV") == "production":
    origins = ["https://yourdomain.com"]
else:
    origins = ["*"]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

CORS Headers Added:
  Access-Control-Allow-Origin: https://yourdomain.com
  Access-Control-Allow-Methods: GET, POST, PUT, DELETE
  Access-Control-Allow-Headers: Authorization, Content-Type
  Access-Control-Allow-Credentials: true
  Access-Control-Max-Age: 3600
""")

## 🗜️ Compression Middleware

In [ ]:
print("""
from fastapi import FastAPI
from fastapi.middleware.gzip import GZipMiddleware

app = FastAPI()

# Add GZip compression
app.add_middleware(
    GZipMiddleware,
    minimum_size=1000  # Only compress responses > 1KB
)

@app.get("/api/large-response")
async def large_response():
    # Return large response
    return {"data": ["item" for _ in range(1000)]}

Response headers:
  Content-Encoding: gzip
  Content-Length: 500  (compressed from 5000)

✅ Benefits:
  - Reduce bandwidth
  - Faster responses
  - Lower costs

⚠️ Tradeoffs:
  - CPU overhead
  - Not needed for small responses
  - Already compressed data (images, videos)
""")

## 🔒 Security Headers Middleware

In [ ]:
from fastapi import Request

@app.middleware("http")
async def add_security_headers(request: Request, call_next):
    """Add security headers to all responses."""
    
    response = await call_next(request)
    
    # Prevent MIME type sniffing
    response.headers['X-Content-Type-Options'] = 'nosniff'
    
    # Prevent clickjacking
    response.headers['X-Frame-Options'] = 'DENY'
    
    # Enable XSS protection
    response.headers['X-XSS-Protection'] = '1; mode=block'
    
    # Strict transport security (HTTPS only)
    response.headers['Strict-Transport-Security'] = 'max-age=31536000; includeSubDomains'
    
    # Content Security Policy
    response.headers['Content-Security-Policy'] = "default-src 'self'"
    
    # Referrer policy
    response.headers['Referrer-Policy'] = 'no-referrer'
    
    # Permissions policy
    response.headers['Permissions-Policy'] = 'geolocation=(), microphone=()'
    
    return response

print("🔒 Security Headers")
print("\nHeaders added:")
print("  X-Content-Type-Options: nosniff")
print("  X-Frame-Options: DENY")
print("  X-XSS-Protection: 1; mode=block")
print("  Strict-Transport-Security: max-age=31536000")
print("  Content-Security-Policy: default-src 'self'")

## 🚨 Error Handling Middleware

In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse
import traceback

@app.middleware("http")
async def catch_exceptions_middleware(request: Request, call_next):
    """Catch and log all exceptions."""
    
    try:
        response = await call_next(request)
        return response
    
    except Exception as e:
        # Log error with details
        logger.error(
            f"Unhandled exception: {str(e)}\n"
            f"Path: {request.url.path}\n"
            f"Method: {request.method}\n"
            f"Traceback: {traceback.format_exc()}"
        )
        
        # Return generic error (don't expose internals)
        return JSONResponse(
            status_code=500,
            content={
                "error": "Internal server error",
                "message": "Something went wrong. Please try again."
            }
        )

print("🚨 Error Handling Middleware")
print("\nCatches all exceptions and:")
print("  1. Logs detailed error")
print("  2. Returns generic 500 error")
print("  3. Prevents app crashes")

## 🏗️ Complete Middleware Stack

In [ ]:
print("""
# Production-ready middleware stack

from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
from fastapi.middleware.trustedhost import TrustedHostMiddleware
import time
import logging
import uuid

app = FastAPI(title="Production API")
logger = logging.getLogger(__name__)

# 1. Trusted hosts (security)
app.add_middleware(
    TrustedHostMiddleware,
    allowed_hosts=["yourdomain.com", "*.yourdomain.com"]
)

# 2. CORS (before routes)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://yourdomain.com"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 3. Compression
app.add_middleware(
    GZipMiddleware,
    minimum_size=1000
)

# 4. Request ID
@app.middleware("http")
async def add_request_id(request: Request, call_next):
    request_id = str(uuid.uuid4())
    request.state.request_id = request_id
    
    response = await call_next(request)
    response.headers['X-Request-ID'] = request_id
    
    return response

# 5. Timing
@app.middleware("http")
async def add_timing(request: Request, call_next):
    start_time = time.time()
    
    response = await call_next(request)
    
    process_time = time.time() - start_time
    response.headers['X-Process-Time'] = f"{process_time:.4f}"
    
    return response

# 6. Logging
@app.middleware("http")
async def log_requests(request: Request, call_next):
    request_id = request.state.request_id
    
    logger.info(
        f"[{request_id}] {request.method} {request.url.path} "
        f"from {request.client.host}"
    )
    
    start_time = time.time()
    response = await call_next(request)
    duration = time.time() - start_time
    
    logger.info(
        f"[{request_id}] {response.status_code} "
        f"in {duration*1000:.2f}ms"
    )
    
    return response

# 7. Security headers
@app.middleware("http")
async def add_security_headers(request: Request, call_next):
    response = await call_next(request)
    
    response.headers['X-Content-Type-Options'] = 'nosniff'
    response.headers['X-Frame-Options'] = 'DENY'
    response.headers['X-XSS-Protection'] = '1; mode=block'
    
    return response

# 8. Error handling (last)
@app.middleware("http")
async def catch_exceptions(request: Request, call_next):
    try:
        return await call_next(request)
    except Exception as e:
        logger.exception(f"Unhandled exception: {e}")
        return JSONResponse(
            status_code=500,
            content={"error": "Internal server error"}
        )

@app.get("/")
async def root():
    return {"message": "Hello World"}

# Middleware execution order:
# Request:  8 → 7 → 6 → 5 → 4 → 3 → 2 → 1 → Handler
# Response: Handler → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8
""")

## ✅ Summary

### Middleware Types:

**1. Logging**
```python
@app.middleware("http")
async def log_requests(request: Request, call_next):
    logger.info(f"{request.method} {request.url.path}")
    response = await call_next(request)
    return response
```

**2. Timing**
```python
@app.middleware("http")
async def add_timing(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    response.headers['X-Process-Time'] = str(time.time() - start)
    return response
```

**3. CORS**
```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://yourdomain.com"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
```

**4. Compression**
```python
app.add_middleware(
    GZipMiddleware,
    minimum_size=1000
)
```

**5. Security**
```python
@app.middleware("http")
async def add_security_headers(request, call_next):
    response = await call_next(request)
    response.headers['X-Content-Type-Options'] = 'nosniff'
    response.headers['X-Frame-Options'] = 'DENY'
    return response
```

### Best Practices:

**1. Order Matters**
```python
# Correct order:
1. TrustedHostMiddleware  (security first)
2. CORSMiddleware
3. GZipMiddleware
4. Custom middleware
5. Error handling (last)
```

**2. Use Request State**
```python
# Store data for other middleware
request.state.request_id = uuid.uuid4()
request.state.user = get_user(request)
```

**3. Handle Errors**
```python
@app.middleware("http")
async def middleware(request, call_next):
    try:
        response = await call_next(request)
        return response
    except Exception as e:
        logger.exception(e)
        return error_response()
```

**4. Keep It Fast**
```python
# ❌ BAD - Slow middleware
@app.middleware("http")
async def slow(request, call_next):
    await slow_db_query()  # Blocks all requests!
    return await call_next(request)

# ✅ GOOD - Fast middleware
@app.middleware("http")
async def fast(request, call_next):
    request.state.start_time = time.time()
    return await call_next(request)
```

**5. Environment-Specific**
```python
import os

if os.getenv("ENV") == "production":
    # Strict CORS
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["https://yourdomain.com"]
    )
else:
    # Permissive for development
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"]
    )
```

### Common Patterns:

**Request ID + Logging:**
```python
request.state.request_id = str(uuid.uuid4())
logger.info(f"[{request.state.request_id}] Processing request")
```

**Timing + Alerting:**
```python
if process_time > 1.0:
    alert(f"Slow request: {request.url.path}")
```

**Auth + Rate Limiting:**
```python
user = get_user(request)
if not rate_limiter.allow(user.id):
    raise HTTPException(429)
```

### Congratulations! 🎉

You've completed the **Production APIs** module!

**What you learned:**
- FastAPI basics
- Streaming responses
- Error handling
- Authentication
- Rate limiting
- Middleware

### Next: `09_caching/01_response_caching.ipynb`